# Experimento — Limitação do Perceptron com XOR

**Aula 1 · 11/11 · Inteligência Artificial aplicada à Engenharia Elétrica**

## Antes de começar

### Objetivo
Avaliar por que um único neurônio linear não resolve a lógica XOR, usando duas entradas binárias sem unidade.

### Pré-requisitos
Nenhum notebook anterior. Execute este arquivo de forma independente em uma sessão nova do Google Colab.

### Como executar
1. Abra o link público deste experimento no Colab: [Abrir Limitação do Perceptron com XOR](https://colab.research.google.com/github/Furriel/code-library/blob/main/cursos/ia/redes-neurais/perceptron-xor.ipynb).
2. No menu **Ambiente de execução → Executar tudo**, rode as células de cima para baixo.
3. Faça a conta à mão antes de verificar a célula numérica.
4. Altere **somente o parâmetro sinalizado** no experimento guiado e execute tudo novamente.
5. Compare os gráficos e registre as conclusões; você não precisa enviar o notebook nem utilizar GitHub.

### Dados e parâmetros
A contradição geométrica, os erros que não desaparecem e o efeito da taxa de aprendizagem sem prometer convergência.

O ambiente usa NumPy e Matplotlib disponíveis no Colab. O compilador C é opcional onde houver exemplo de referência.


## Cálculo manual — um passo no XOR

Para `x=(1,1)`, `w=(0,2; 0,2)`, `b=-0,1`, `d=0` e `η=0,2`, calcule `z = 0,3`, `y = 1`, `e = -1`; a atualização produz `w'=(0;0)` e `b'=-0,3`. O exemplo mostra um passo local, **não** prova convergência para toda a tabela.


In [ ]:
x=(1,1); w=[0.2,0.2]; b=-0.1; d=0; eta=0.2
z=sum(a*b_ for a,b_ in zip(x,w))+b
y=int(z>=0); e=d-y
novo_w=[a+eta*e*xx for a,xx in zip(w,x)]; novo_b=b+eta*e
assert abs(z-0.3)<1e-12 and y==1 and e==-1
assert all(abs(v)<1e-12 for v in novo_w) and abs(novo_b+0.3)<1e-12
print("CÁLCULO MANUAL CONFERIDO: z=0,3; y=1; e=-1; w'=(0,0); b'=-0,3")


In [ ]:
import json, math, shutil, subprocess
import numpy as np
import matplotlib.pyplot as plt

GCC = shutil.which("gcc") is not None
print("GCC disponível:", GCC)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

class LCG:
    # Mesmo gerador da página: s <- (1664525*s + 1013904223) mod 2^32; u = s / 2^32.
    def __init__(self, semente):
        self.s = semente % 2**32
    def proximo(self):
        self.s = (self.s * 1664525 + 1013904223) % 2**32
        return self.s / 4294967296
    def inteiro(self, n):
        return int(self.proximo() * n)

In [ ]:
ENTRADAS = [(0, 0), (0, 1), (1, 0), (1, 1)]
TABELAS = {"OR": [0, 1, 1, 1], "AND": [0, 0, 0, 1], "XOR": [0, 1, 1, 0]}

def treinar_perceptron(tabela, w0=(0.0, 0.0), b0=0.0, eta=0.2, max_epocas=100):
    w, b = list(w0), b0
    passos, epocas, vistos = [(w[0], w[1], b)], [], {}
    for ep in range(1, max_epocas + 1):
        erros = 0
        for x, d in zip(ENTRADAS, TABELAS[tabela]):
            z = b
            for i in range(2):
                z += w[i] * x[i]
            e = d - (1 if z >= 0 else 0)
            if e != 0:
                erros += 1
                for i in range(2):
                    w[i] += eta * e * x[i]
                b += eta * e
            passos.append((w[0], w[1], b))
        epocas.append((ep, erros, w[0], w[1], b))
        if erros == 0:
            return "convergiu", passos, epocas
        chave = tuple(round(v, 9) for v in (w[0], w[1], b))
        if chave in vistos:
            return f"ciclo (repete a época {vistos[chave]})", passos, epocas
        vistos[chave] = ep
    return "limite", passos, epocas



## 1. XOR: a limitação do perceptron simples

XOR exige $(0,1)$ e $(1,0)$ de um lado e $(0,0)$ e $(1,1)$ do outro. Somando as desigualdades: $w_1+w_2+2b \ge 0$ e $w_1+w_2+2b<0$ — contradição. Nenhuma reta separa os dados; o treino entra em ciclo.

In [ ]:
status_xor, passos_xor, epocas_xor = treinar_perceptron("XOR", max_epocas=50)
print("XOR:", status_xor, "após", len(epocas_xor), "épocas")
assert not status_xor.startswith("convergiu")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar([e[0] for e in epocas_xor], [e[1] for e in epocas_xor], color="#b42318")
ax[0].set(title="XOR: erros por época nunca chegam a zero", xlabel="época", ylabel="erros (de 4)", ylim=(0, 4))
p = np.array(passos_xor)
for k, nome in enumerate(["w₁", "w₂", "b"]):
    ax[1].step(range(len(p)), p[:, k], where="post", label=nome)
ax[1].set(title="Parâmetros oscilam em ciclo", xlabel="passo", ylabel="valor")
ax[1].legend(); plt.tight_layout(); plt.show()

## Experimentos propostos — Experimento guiado: XOR antes × depois

Altere apenas `ETA_XOR_ALTERADA` para verificar o efeito da taxa mantendo semente, exemplos e condições iniciais. **Não espere convergência**, independentemente da taxa. Registre curvas e interpretação no relatório único do Moodle.


In [ ]:
ETA_XOR_ORIGINAL = 0.2
ETA_XOR_ALTERADA = 0.5  # ALTERE SOMENTE ESTE VALOR (0 < eta <= 1)
assert 0 < ETA_XOR_ALTERADA <= 1
s0, _, e0 = treinar_perceptron("XOR", eta=ETA_XOR_ORIGINAL, max_epocas=50)
s1, _, e1 = treinar_perceptron("XOR", eta=ETA_XOR_ALTERADA, max_epocas=50)
print("ANTES × DEPOIS — Perceptron XOR")
for taxa, st, epocas in [(ETA_XOR_ORIGINAL,s0,e0),(ETA_XOR_ALTERADA,s1,e1)]:
    print(f"eta={taxa:.2f}: {st}; épocas={len(epocas)}; erros na última época={epocas[-1][1]}")
assert s0 != "convergiu" and s1 != "convergiu", "XOR não é linearmente separável"
fig,ax=plt.subplots(figsize=(9,4))
ax.step([v[0] for v in e0],[v[1] for v in e0],where="post",marker="o",label="original")
ax.step([v[0] for v in e1],[v[1] for v in e1],where="post",marker="s",label="alterada")
ax.set(xlabel="época",ylabel="erros (de 4)",title="XOR — duas taxas sem convergência"); ax.legend()
plt.tight_layout(); plt.show()
print("INTERPRETAÇÃO AUTOMÁTICA: nenhum cenário convergiu. Diferenças em erros ou duração do ciclo não removem a impossibilidade de separar XOR com uma reta.")
print("Registre a comparação no relatório único do Moodle.")



## Questões de interpretação

Por que nenhuma reta classifica XOR? O erro final varia com a taxa, mas alguma execução convergiu? Registre o argumento matemático, as curvas e as limitações no relatório único do Moodle.

## Conclusão

Este experimento é independente. Use o mesmo conjunto de dados e as mesmas condições iniciais ao comparar parâmetros. **Experimentos propostos:** altere apenas o valor indicado no código e justifique o resultado observado, sem afirmar melhora quando as métricas não a demonstram. Entregue somente um **único relatório final** pelo Moodle, conforme as instruções da disciplina.
